# Cross-Framework Comparison - Softmax Regression

Global imports

In [ ]:
from river import linear_model, optim
from tabulate import tabulate
import os
import importlib.util
import sys
import matplotlib.pyplot as plt

# We use an external library (scikit-learn) to compute the metrics consistently across models
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

Initial environment configuration

In [ ]:
# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [ ]:
DEFAULT_LR = 0.01
DEFAULT_L2 = 0.0
MAX_INSTANCES = 100000
SEED = 42

Dynamically load the custom SoftmaxRegression over the installed capymoa package

In [ ]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_softmax_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._softmax_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._softmax_regression"] = module
spec.loader.exec_module(module)
SoftmaxRegression = module.SoftmaxRegression

Global functions

In [ ]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if i > MAX_INSTANCES:
            break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}
        x["bias"] = 1.0 # add a constant feature to simulate the bias/intercept, since River's SoftmaxRegression has no explicit bias term.

        # label
        y = instance.y_index

        data.append((x, y))

    return data

In [ ]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, l2=DEFAULT_L2):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, l2)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, l2)

    table = []
    for key in ["Accuracy", "F1", "Precision", "Recall"]:
        capy = capyMoaResults["metrics"][key]
        river = riverResults["metrics"][key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

    # confusion matrices

    labels = sorted(set(capyMoaResults["y_true"]) | set(riverResults["y_true"]))

    _plot_confusion_matrices(
        capyMoaResults["y_true"],
        capyMoaResults["y_pred"],
        riverResults["y_true"],
        riverResults["y_pred"],
        labels
    )

def _evaluateStreamOnCapyMoa(stream, lr, l2):
    softmax_reg_capymoa = SoftmaxRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        l2_penalty=l2
    )

    y_true = []
    y_pred = []
    default_label = None

    # prequential evaluation
    for i, instance in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        true_label = int(instance.y_index)

        if default_label is None:
            default_label = true_label

        # test
        votes = softmax_reg_capymoa.predict_proba(instance)

        if votes is None or len(votes) == 0:
            pred = default_label
        else:
            pred = max(range(len(votes)), key=lambda j: votes[j])

        y_true.append(true_label)
        y_pred.append(pred)

        # train
        softmax_reg_capymoa.train(instance)

    return {
        "metrics": _compute_sklearn_metrics(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
    }

def _evaluateStreamOnRiver(stream, lr, l2):
    softmax_reg_river = linear_model.SoftmaxRegression(
        optimizer=optim.SGD(lr),
        l2=l2
    )

    y_true = []
    y_pred = []
    default_label = None

    # prequential evaluation
    for i, (x, y) in enumerate(stream):
        if i >= MAX_INSTANCES:
            break

        if default_label is None:
            default_label = y

        pred = softmax_reg_river.predict_one(x)

        if pred is None:
            pred = default_label

        y_true.append(y)
        y_pred.append(pred)

        softmax_reg_river.learn_one(x, y)

    return {
        "metrics": _compute_sklearn_metrics(y_true, y_pred),
        "y_true": y_true,
        "y_pred": y_pred,
    }

def _compute_sklearn_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
    }

def _plot_confusion_matrices(
    y_true_capy,
    y_pred_capy,
    y_true_river,
    y_pred_river,
    labels
):
    cm_capy = confusion_matrix(y_true_capy, y_pred_capy, labels=labels)
    cm_river = confusion_matrix(y_true_river, y_pred_river, labels=labels)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    disp1 = ConfusionMatrixDisplay(cm_capy, display_labels=labels)
    disp1.plot(ax=axes[0], cmap="Oranges", colorbar=False, values_format="d")
    axes[0].set_title("CapyMOA")

    disp2 = ConfusionMatrixDisplay(cm_river, display_labels=labels)
    disp2.plot(ax=axes[1], cmap="Oranges", colorbar=False, values_format="d")
    axes[1].set_title("River")

    plt.tight_layout()
    plt.show()

## Electricity dataset (2 classes)

In [ ]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)

## RandomRBFGenerator

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)

## WaveformGenerator

In [ ]:
from capymoa.stream.generator import WaveformGenerator

def make_stream():
    return WaveformGenerator(instance_random_seed=SEED)

evaluateStream(make_stream)

## RTG_2abrupt dataset

In [ ]:
from capymoa.datasets import RTG_2abrupt

evaluateStream(RTG_2abrupt)

## RandomTreeGenerator

In [ ]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=5,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)

## RandomRBFGeneratorDrift

In [ ]:
from capymoa.stream.generator import RandomRBFGeneratorDrift

def make_stream():
    return RandomRBFGeneratorDrift(
        number_of_classes=5,
        number_of_attributes=10,
        number_of_centroids=50,
        number_of_drifting_centroids=2,
        magnitude_of_change=0.5,
    )

evaluateStream(make_stream)

## DriftStream dataset generator
(reference: 04_drift_streams capymoa's tutorial)

In [ ]:
from capymoa.stream.drift import DriftStream, AbruptDrift, GradualDrift
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return DriftStream(
        stream = [
            RandomTreeGenerator(num_classes=5, instance_random_seed=1, tree_random_seed=1), # concept A
            AbruptDrift(position=5000),
            RandomTreeGenerator(num_classes=5, instance_random_seed=2, tree_random_seed=2), # concept B
            GradualDrift(position=10000, width=3000),
            RandomTreeGenerator(num_classes=5, instance_random_seed=3, tree_random_seed=3), # concept C
            AbruptDrift(position=14000),
            RandomTreeGenerator(num_classes=5, instance_random_seed=1, tree_random_seed=1), # concept A again
        ]
    )

evaluateStream(make_stream)

## RandomRBFGenerator (changed model parameters)

### L2

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream, l2=0.01)

### Learning rate

In [ ]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=7,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream, lr=0.1)